In [1]:
%pip install networkx

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 20.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from agents.db import get_connection
from agents.planner import planner_agent

conn = get_connection()

In [3]:
print(planner_agent(conn, "Harvard", "Back Bay"))
print(planner_agent(conn, "Harvard", "Back Bay", avoid=["place-dwnxg"]))

{'ok': True, 'path': ['place-harsq', 'place-cntsq', 'place-knncl', 'place-chmnl', 'place-pktrm', 'place-dwnxg', 'place-chncl', 'place-tumnl', 'place-bbsta'], 'stations': ['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'Chinatown', 'Tufts Medical Center', 'Back Bay'], 'transfers': ['Downtown Crossing']}
{'ok': False, 'reason': 'No route found avoiding those stations'}


In [4]:
print(planner_agent(conn, "Harvard", "Haymarket")["stations"])
print(planner_agent(conn, "Harvard", "Haymarket", avoid=["place-gover"])["stations"])

['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Government Center', 'Haymarket']
['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'State', 'Haymarket']


In [5]:
tests = [
    # (origin, destination, avoid, what you should see)
    ("Harvard", "Back Bay", None, "Red to Downtown Crossing, then Orange"),
    ("Wonderland", "Forest Hills", None, "Blue to State, then Orange"),
    ("Alewife", "Ashmont", None, "Red Line only, no transfers"),
    ("Ashmont", "Braintree", None, "must go through JFK/UMass (branch fix)"),
    ("Harvard", "Haymarket", ["place-gover"], "avoids Government Center"),
    ("harvard", "KENDALL/MIT", None, "works with any capitalization"),
    ("Hogwarts", "Harvard", None, "should fail: station not found"),
    ("Boylston", "Harvard", None, "should fail: not wheelchair accessible"),
]

for origin, dest, avoid, expected in tests:
    result = planner_agent(conn, origin, dest, avoid=avoid)
    print(f"{origin} -> {dest}" + (f"  (avoid {avoid})" if avoid else ""))
    print(f"  Expected: {expected}")
    if result["ok"]:
        print(f"  Route:     {' -> '.join(result['stations'])}")
        print(f"  Transfers: {result['transfers'] or 'none'}")
    else:
        print(f"  Failed:    {result['reason']}")
    print()

Harvard -> Back Bay
  Expected: Red to Downtown Crossing, then Orange
  Route:     Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay
  Transfers: ['Downtown Crossing']

Wonderland -> Forest Hills
  Expected: Blue to State, then Orange
  Route:     Wonderland -> Revere Beach -> Beachmont -> Suffolk Downs -> Orient Heights -> Wood Island -> Airport -> Maverick -> Aquarium -> State -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay -> Massachusetts Avenue -> Ruggles -> Roxbury Crossing -> Jackson Square -> Stony Brook -> Green Street -> Forest Hills
  Transfers: ['State']

Alewife -> Ashmont
  Expected: Red Line only, no transfers
  Route:     Alewife -> Davis -> Porter -> Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> South Station -> Broadway -> Andrew -> JFK/UMass -> Savin Hill -> Fields Corner -> Shawmut -> Ashmont
  Transfers: none

Ashmont -